> ### ⚠️ Select the **`Python 3 (croprow)`** kernel first
> This notebook runs in the isolated croprow env (Python **3.11**, OpenCV). If the first cell throws `ModuleNotFoundError: No module named 'cv2'`, the wrong interpreter is selected.
>
> **VSCode:** click **Select Kernel** (top-right) → **Jupyter Kernel** → **`Python 3 (croprow)`**, or **Python Environments... → Enter interpreter path...** and paste `croprow\.venv\Scripts\python.exe`.
>
> Do **not** use the repo-root `.venv` — that is the potato backend (Python 3.13, no cv2 by design). `croprow_disease` shares the `croprow` env; it needs no venv of its own.

# 02 — Verify the derived labels visually

Two checks, because there are two derivations to trust:

1. **Boxes** — draw them on random frames and confirm they sit on plants.
2. **Classes** — the part that is a heuristic. The diagnostic strip sorts instances by health score and shows the extremes, so you can see *what the rule is actually reacting to*.

That second check is not decoration. Run on LettuceMOTS with the quality gate disabled, it shows the lowest-scoring instances are motion-blurred and shadowed captures rather than brown plants — which is how the gate in `health.py` got its thresholds. Do the same on your own frames before trusting a single class number.

Boxes and classes are derived on the fly from the source polygons, so this is an honest check of the rule, independent of whether 01 has been run.

**Green box = healthy, orange-red = unhealthy.** The number on each box is the health score (fraction of canopy pixels reading as vigorous green).

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# === CONFIG: the ONLY place to set the dataset location ===================
LETTUCE_ROOT_DEFAULT = r"D:\croprow_dataset\LettuceMOTS"
LETTUCE_ROOT = os.environ.get("LETTUCE_ROOT", LETTUCE_ROOT_DEFAULT)
PARAMS = HealthParams()

N_SHOW = 12      # frames to draw
N_CROPS = 10     # per-row instance crops in the diagnostic strip
SEED = 7
# ==========================================================================

root = U.resolve_lettuce_root(LETTUCE_ROOT)
print("LETTUCE_ROOT :", root)

%matplotlib inline
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2

## Sample frames across all labeled sequences

In [ ]:
all_frames = []
for s in U.labeled_sequences(root):
    all_frames.extend(U.frame_pairs(root, s))
print("total labeled frames:", len(all_frames))

rng = random.Random(SEED)
sample = rng.sample(all_frames, min(N_SHOW, len(all_frames)))
print("showing", len(sample), "frames")

## A) Boxes + classes on whole frames

In [ ]:
cols = 3
rows = (len(sample) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.6 * rows))
axes = np.atleast_1d(axes).ravel()
for ax, (img_path, lab_path) in zip(axes, sample):
    bgr = U.imread_bgr(img_path)
    inst = U.instances_for_frame(bgr, lab_path, PARAMS)
    drawn = U.draw_instances(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), inst, rgb=True)
    n_h = sum(1 for i in inst if i["cls"] == U.HEALTHY)
    n_u = len(inst) - n_h
    ax.imshow(drawn)
    ax.set_title(f"{img_path.parent.name}/{img_path.stem}  |  "
                 f"{n_h} healthy, {n_u} unhealthy", fontsize=8)
    ax.axis("off")
for ax in axes[len(sample):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## B) Diagnostic strip — what is the rule reacting to?

Instance crops sorted by health score: the lowest scorers (what the rule calls unhealthy), the median, and the highest. Each is annotated `score / sharpness`.

**Read the top row carefully.** If those crops are blurred, dark or smeared rather than brown, the rule is measuring image quality and not plant health — raise `min_sharpness` / `min_mean_value` in `PARAMS` until only genuinely off-colour plants remain.

In [ ]:
scan = rng.sample(all_frames, min(120, len(all_frames)))
found = []
for img_path, lab_path in scan:
    bgr = U.imread_bgr(img_path)
    for i in U.instances_for_frame(bgr, lab_path, PARAMS):
        found.append((i, img_path))
found.sort(key=lambda t: t[0]["score"])
print(f"scanned {len(scan)} frames -> {len(found)} instances")
n_gated = sum(1 for i, _ in found if not i["judged"])
print(f"quality-gated (undecidable, defaulted to healthy): {n_gated} "
      f"({100 * n_gated / max(len(found), 1):.1f}%)")

CELL = 120
def crop_cell(inst, img_path):
    bgr = U.imread_bgr(img_path)
    h, w = bgr.shape[:2]
    x0 = max(0, int((inst["cx"] - inst["w"] / 2) * w))
    x1 = min(w, int((inst["cx"] + inst["w"] / 2) * w))
    y0 = max(0, int((inst["cy"] - inst["h"] / 2) * h))
    y1 = min(h, int((inst["cy"] + inst["h"] / 2) * h))
    c = bgr[y0:y1, x0:x1]
    if c.size == 0:
        c = np.zeros((10, 10, 3), np.uint8)
    c = cv2.resize(c, (CELL, CELL), interpolation=cv2.INTER_AREA)
    tag = f"{inst['score']:.2f} s{inst['sharpness']:.0f}"
    if not inst["judged"]:
        tag += " GATE"
    cv2.putText(c, tag, (3, CELL - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.36,
                (0, 255, 255), 1, cv2.LINE_AA)
    return cv2.cvtColor(c, cv2.COLOR_BGR2RGB)

def strip(items):
    cells = [crop_cell(i, p) for i, p in items[:N_CROPS]]
    while len(cells) < N_CROPS:
        cells.append(np.zeros((CELL, CELL, 3), np.uint8))
    return np.hstack(cells)

mid = len(found) // 2
bands = [("LOWEST score", found[:N_CROPS]),
         ("MEDIAN score", found[mid - N_CROPS // 2: mid + N_CROPS // 2]),
         ("HIGHEST score", found[-N_CROPS:])]
fig, axes = plt.subplots(len(bands), 1, figsize=(16, 2.2 * len(bands)))
for ax, (title, items) in zip(np.atleast_1d(axes), bands):
    ax.imshow(strip(items))
    ax.set_title(title, fontsize=10, loc="left")
    ax.axis("off")
plt.tight_layout()
plt.show()

## C) Score histogram + class balance

In [ ]:
scores = np.array([i["score"] for i, _ in found
                   if i["judged"] and np.isfinite(i["score"])])
n_h = sum(1 for i, _ in found if i["cls"] == U.HEALTHY)
n_u = len(found) - n_h

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(scores, bins=40, range=(0, 1), color="#2e7d32")
ax.axvline(PARAMS.green_frac_threshold, color="#d84315", ls="--",
           label=f"threshold {PARAMS.green_frac_threshold}")
ax.set_xlabel("health score (fraction of canopy pixels reading as vigorous green)")
ax.set_ylabel("instances")
ax.set_title(f"judged instances: {len(scores)}   |   "
             f"healthy {n_h} / unhealthy {n_u}")
ax.legend()
plt.tight_layout()
plt.show()

print(U.check_class_balance({"healthy": n_h, "unhealthy": n_u})["message"])